# 05 - Sales Forecasting

Forecast monthly sales using Prophet. Given findings so far:
- Bulk orders (1% of transactions, 37.6% of revenue) are lumpy and distort time series - forecast retail and bulk separately
- Germany and Poland are structurally different markets - forecast separately by country

We'll build: Germany-Retail, Germany-Bulk, Poland-Retail, Poland-Bulk (and an overall total for comparison).

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from eda_utils import load_clean_data, save_fig
from segmentation import filter_valid_sales, add_order_type
from forecasting import (
    prepare_monthly_series, fit_prophet, forecast_future,
    plot_forecast, plot_components, train_test_split_series,
    evaluate_forecast, plot_actual_vs_predicted
)

In [ ]:
df = load_clean_data("../data/processed/cleaned_data.csv")
df = filter_valid_sales(df)
df = add_order_type(df, bulk_quantile=0.99)
df.head()

## 1. Overall sales forecast (baseline, for comparison)

In [ ]:
overall_series = prepare_monthly_series(df)
print(f"Series length: {len(overall_series)} months")
overall_series.tail()

In [ ]:
train, test = train_test_split_series(overall_series, test_periods=6)
model_overall = fit_prophet(train)
forecast_overall = forecast_future(model_overall, periods=6 + 12)  # cover test period + 12 future months

metrics = evaluate_forecast(model_overall, test)
print({k: v for k, v in metrics.items() if k != 'comparison'})

In [ ]:
fig = plot_actual_vs_predicted(metrics['comparison'], title='Overall - Actual vs Predicted (test period)')
save_fig(fig, '18_overall_actual_vs_predicted', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_forecast(model_overall, forecast_overall, title='Overall Sales Forecast')
save_fig(fig, '19_overall_forecast', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_components(model_overall, forecast_overall)
save_fig(fig, '20_overall_components', path='../reports/figures')
plt.show()

## 2. Forecast by Country x Order Type (Retail / Bulk)

We'll loop through the 4 segments and fit a separate model for each.

In [ ]:
segments = {
    'Germany - Retail': {'country': 'Germany', 'order_type': 'retail'},
    'Germany - Bulk': {'country': 'Germany', 'order_type': 'bulk'},
    'Poland - Retail': {'country': 'Poland', 'order_type': 'retail'},
    'Poland - Bulk': {'country': 'Poland', 'order_type': 'bulk'},
}

results = {}

for name, filt in segments.items():
    series = prepare_monthly_series(df, group_filter=filt)
    print(f"{name}: {len(series)} months, total sales = {series['y'].sum():,.0f}")
    results[name] = {'series': series}

### Diagnostic: check for short/sparse series before fitting

Poland's MAPE blew up (900%+) likely due to a short or sparse series confusing Prophet's trend/changepoint detection.
Inspect each series' length, zero-months, and min/max before fitting.

In [ ]:
for name, res in results.items():
    s = res['series']
    print(f"\n{name}")
    print(f"  n_months: {len(s)}")
    print(f"  zero/near-zero months: {(s['y'] < s['y'].mean() * 0.05).sum()}")
    print(f"  min: {s['y'].min():,.0f}, max: {s['y'].max():,.0f}, mean: {s['y'].mean():,.0f}")
    print(s.tail(8).to_string(index=False))

In [ ]:
# Fit, evaluate, and forecast for each segment
TEST_PERIODS = 6
FUTURE_PERIODS = 12

for name, res in results.items():
    series = res['series']
    train, test = train_test_split_series(series, test_periods=TEST_PERIODS)

    # More conservative settings for shorter/sparser series (e.g. Poland) to avoid
    # runaway trend extrapolation. Lower changepoint_prior_scale = less flexible trend.
    n_years = len(train) / 12
    use_yearly_seasonality = n_years >= 2  # need >=2 full years to estimate yearly seasonality
    cps = 0.05 if n_years >= 2 else 0.01

    model = fit_prophet(train, yearly_seasonality=use_yearly_seasonality, changepoint_prior_scale=cps)
    forecast = forecast_future(model, periods=TEST_PERIODS + FUTURE_PERIODS)

    # Sales cannot be negative - clip predictions at 0
    for col in ['yhat', 'yhat_lower', 'yhat_upper']:
        forecast[col] = forecast[col].clip(lower=0)

    metrics = evaluate_forecast(model, test)

    results[name].update({
        'model': model,
        'forecast': forecast,
        'metrics': metrics,
    })

    print(f"\n{name}")
    print({k: round(v, 2) if isinstance(v, float) else v for k, v in metrics.items() if k != 'comparison'})

## 3. Visualize each segment

In [ ]:
for i, (name, res) in enumerate(results.items()):
    fig = plot_actual_vs_predicted(res['metrics']['comparison'], title=f'{name} - Actual vs Predicted (test)')
    save_fig(fig, f'21_{i}_actual_vs_predicted_{name.replace(" ", "_").replace("-", "")}', path='../reports/figures')
    plt.show()

In [ ]:
for i, (name, res) in enumerate(results.items()):
    fig = plot_forecast(res['model'], res['forecast'], title=f'{name} - Forecast')
    save_fig(fig, f'22_{i}_forecast_{name.replace(" ", "_").replace("-", "")}', path='../reports/figures')
    plt.show()

## 4. Summary table - accuracy across segments

In [ ]:
summary_rows = []
for name, res in results.items():
    m = res['metrics']
    summary_rows.append({
        'segment': name,
        'MAE': m['MAE'],
        'RMSE': m['RMSE'],
        'MAPE_%': m['MAPE'],
        'MdAPE_%': m['MdAPE'],
        'avg_monthly_sales': res['series']['y'].mean(),
    })

accuracy_summary = pd.DataFrame(summary_rows)
accuracy_summary

## 5. Next 12-month outlook per segment

In [ ]:
outlook_rows = []
for name, res in results.items():
    fc = res['forecast']
    last_actual_date = res['series']['ds'].max()
    future_only = fc[fc['ds'] > last_actual_date]
    next_12_total = future_only.head(12)['yhat'].sum()
    last_12_actual = res['series'].tail(12)['y'].sum()

    outlook_rows.append({
        'segment': name,
        'last_12_months_actual': last_12_actual,
        'next_12_months_forecast': next_12_total,
        'change_pct': (next_12_total - last_12_actual) / last_12_actual * 100 if last_12_actual else None,
    })

outlook = pd.DataFrame(outlook_rows)
outlook

## Key Observations (fill in after running)

- Which segment is easiest/hardest to forecast (lowest/highest MAPE)? Bulk segments likely have high MAPE due to lumpiness.
- Overall 12-month outlook: growth or decline by segment?
- Does Poland's forecast suggest continued stagnation (consistent with 0% growth from segmentation)?
- Recommendation: how should bulk-order unpredictability be communicated to stakeholders (e.g. confidence intervals, separate 'large deal pipeline' tracking vs core retail forecast)?